In [1]:
import pandas as pd
import geopandas as gpd

from shared_utils import rt_utils, catalog_utils, rt_dates, webmap_utils

from calitp_data_analysis import get_fs, geography_utils
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
gcsgp = GCSGeoPandas()
from segment_speed_utils import helpers, time_series_utils, gtfs_schedule_wrangling, corridor_analysis
from segment_speed_utils.project_vars import SCHED_GCS, SEGMENT_GCS, GTFS_DATA_DICT
import numpy as np
import datetime as dt

ADHOC_PATH = "gs://calitp-analytics-data/data-analyses/rt_segment_speeds/adhoc/"

c:\Users\s156485\Documents\data-analyses\.venv\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# Run SCCP/LPP Level of Transit Delay Metric using newer corridor tools

## Example narrative, 16mph benchmark

> The speed-based metric for the corridor provided is [insert number] minutes. This is the sum of delays for each trip traversing the corridor as compared to a reference speed of 16 miles per hour. To further explain, we took each corridor trip that we have data for and calculated the hypothetical time it would take for that trip to traverse the corridor at a speed of 16 mph. The difference between the actual time it took for the trip to traverse the corridor and that hypothetical time is the speed-based delay for that trip, and we summed those delays to create the metric for a day ([insert full date]).
>
> This metric is intended to provide a more consistent basis for comparison independent of scheduling practices.
>
> To calculate which routes we consider to be along the corridor, we first filtered to routes stopping within the corridor. We then filtered our data for those trips to the subset of each trip from the last stop before entering the corridor to the first stop after leaving the corridor.
>
> Here is a list of routes included in our analysis: [insert list of routes here]
>
>Additionally, attached is a map of the corridor and transit routes within it.


## Example narrative, 15% speedup mode

> The speed-based metric for the corridor provided is [insert number] minutes. This is the sum of delays for each trip traversing the corridor assuming a 15% speed increase is possible. The difference between the actual time it took for the trip to traverse the corridor and that hypothetical time is the speed-based delay for that trip, and we summed those delays to create the metric for a day ([insert full date]).
>
> This metric is intended to provide a more consistent basis for comparison independent of scheduling practices.
>
> To calculate which routes we consider to be along the corridor, we first filtered to routes stopping within the corridor. We then filtered our data for those trips to the subset of each trip from the last stop before entering the corridor to the first stop after leaving the corridor.
>
> Here is a list of routes included in our analysis: Union City 2, 4 -> 3, 3 -> 4, AC Transit 56
>
> Additionally, attached is a map of the corridor and transit routes within it.

In [2]:
corridor = gcsgp.read_file(f'{ADHOC_PATH}2026-11-17_sccp_sbline.geojson').assign(corridor_id = 1, corridor_distance_meters = 19000)
#  use short side for distance

In [3]:
analysis_date = rt_dates.DATES['oct2026']

In [4]:
analysis_date

'2026-10-14'

In [5]:
import importlib
importlib.reload(corridor_analysis)

<module 'segment_speed_utils.corridor_analysis' from 'c:\\Users\\s156485\\Documents\\data-analyses\\rt_segment_speeds\\segment_speed_utils\\corridor_analysis.py'>

In [6]:
segment_speeds = corridor_analysis.import_speedmap_segment_speeds(analysis_date)
frequencies = corridor_analysis.get_max_frequencies(segment_speeds)
trip_speeds = corridor_analysis.import_trip_speeds(analysis_date)

2.5 percent of segments have no speed


In [7]:
# feeds = ["Sacramento Schedule", "Elk Grove Schedule"]
# segment_speeds = segment_speeds.query('name.isin(@feeds)')

In [8]:
corridor_trips = corridor_analysis.find_corridor_data(segment_speeds, corridor, trip_speeds,
                                                    #  custom_relevance_meters=100
                                                    )

In [9]:
# exclude SB Line
corridor_trips = corridor_trips[corridor_trips.route_short_name != 'SB Line']


In [10]:
corridor_results = corridor_analysis.analyze_corridor_trips(corridor_trips)

0.1 percent of trips with zero seconds
0.1 percent of trips with speeds > 80mph or < 3mph


In [11]:
corridor_trips.route_short_name.unique()

array(['197', '492', '188', '82', '84', '81', '66', '87', '85', '367',
       '480', '292', '83', '291'], dtype=object)

In [21]:
corridor_trips.name.unique()

array(['Foothill Schedule', 'OmniTrans Schedule'], dtype=object)

In [26]:
def list_routes_by_agency(corridor_trips_df):

    for name in corridor_trips.name.unique():
        print(name + ': ' + ', '.join(corridor_trips[corridor_trips.name == name].route_short_name.unique()))

In [27]:
list_routes_by_agency(corridor_trips)

Foothill Schedule: 197, 492, 188, 480, 292, 291
OmniTrans Schedule: 82, 84, 81, 66, 87, 85, 367, 83


In [14]:
# corridor_analysis.validate_corridor_routes(corridor, corridor_trips)

2

In [15]:
corridor_improvements = corridor_analysis.analyze_corridor_improvements(corridor_results, **{'trip_mph_floor': 16, 'trip_percent_speedup': 15})

median: 14.13mph... percent mode: 0.15


In [16]:
corridor_improvements.head(3)

,trip_instance_key,corridor_meters,corridor_seconds,corridor_speed_mps,corridor_speed_mph,route_short_name,route_id,shape_array_key,shape_id,schedule_gtfs_dataset_key,time_of_day,corridor_id,improved_corridor_speed_mph,intervention_assumption,improved_corridor_speed_mps,improved_corridor_seconds
0,000d28b618cb5ea0468b26866adb4eaa,4243.545750,597.0,7.108117,15.900857,81,81,8893fbf1e4446298a265118645745ab2,41370,2187bd8e29f5c69e2e4402f5d9b59d62,AM Peak,1,18.285986,15.0% trip speed increase,8.174334,519.130435
1,0016c96ec6577355bad233fc2e1682c7,13043.868700,2191.0,5.953386,13.317724,188,20188,104e58306887b7929036b86331127b45,22902_shp-v50-a2,56cfdfc64aafa6efd994e2e5b2c65a7b,PM Peak,1,15.315383,15.0% trip speed increase,6.846394,1905.217391
2,003fb03ebb89683ab70cb874b023cf5a,5845.264082,756.0,7.731831,17.296105,197,10197,9f345ebb03f5afdf068516174674e32d,15776_shp-v4A-a1,56cfdfc64aafa6efd994e2e5b2c65a7b,PM Peak,1,19.890521,15.0% trip speed increase,8.891605,657.391304


In [17]:
summarized = corridor_analysis.summarize_corridor_improvements(corridor_improvements, frequencies, corridor)

c:\Users\s156485\Documents\data-analyses\rt_segment_speeds\segment_speed_utils\corridor_analysis.py:333: FutureWarning: The provided callable <function median at 0x000001E424F57BA0> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  analysis_df = group.agg(


In [18]:
summarized

,corridor_id,schedule_gtfs_dataset_key,intervention_assumption,corridor_miles,corridor_seconds,improved_corridor_seconds,delay_seconds,delay_minutes,minutes_per_mile,route_short_name,route_id,median_corridor_mph,n_trips_daily,trips_hr_sch,trips_per_hr_peak_directional
0,1,2187bd8e29f5c69e2e4402f5d9b59d62,15.0% trip speed increase,11.8,505556.0,439613.9,65942.1,1099.0,93.1,"[81, 84, 367, 87, 66, 82, 83, 85]","[81, 84, 367, 87, 66, 82, 83, 85]",14.8,336,"[1.3, 1.0, 1.0, 1.7, 3.0, 1.3, 2.0, 2.4]",13.7
1,1,56cfdfc64aafa6efd994e2e5b2c65a7b,15.0% trip speed increase,11.8,577653.0,502307.0,75346.0,1255.8,106.4,"[188, 197, 291, 492, 480, 292]","[20188, 10197, 10291, 10492, 20480, 10292]",13.7,448,"[4.0, 1.7, 3.8, 3.0, 3.0, 2.0]",17.5


In [19]:
summarized.delay_minutes.sum()

2354.8